# Byte I/O - Rust

All 40 Rust examples from [docs/io.md](https://platob.github.io/yggdryl/io/), in page order.

Generated by `scripts/build_docs_notebooks.py` from the blocks that
`scripts/check_docs_examples.py` compiles and runs, so every code cell below is
an example that passed. An edit here lives until the next build overwrites it.

The cells are unexecuted and expect the
[evcxr](https://github.com/evcxr/evcxr) kernel. Declare the crate once, in
a cell of your own, before running them:

```rust
:dep yggdryl = { version = "0.1", features = ["parquet", "iceberg"] }
```

In [ ]:
use yggdryl::io::{Buffer, IOBase};

let mut handle = Buffer::new();
handle.pwrite(0, b"symbol,price\n")?;
handle.pwrite(13, b"AAPL,1\n")?;
assert_eq!(handle.size(), 20);

// Two reads at different offsets, in any order: there is no shared cursor.
let mut tail = [0_u8; 4];
handle.pread(13, &mut tail)?;
let mut head = [0_u8; 6];
handle.pread(0, &mut head)?;
assert_eq!(&head, b"symbol");
assert_eq!(&tail, b"AAPL");

## Streamed bytes

In [ ]:
use yggdryl::io::{Buffer, IOBase, IOCursor};

let handle = Buffer::from_bytes(b"0123456789".to_vec());
let chunks = handle
    .pstream_bytes(2, 3)?
    .collect::<yggdryl::Result<Vec<_>>>()?;
assert_eq!(chunks, [b"234".to_vec(), b"567".to_vec(), b"89".to_vec()]);

// The cursor form starts at `tell` and advances only when bytes are yielded.
let mut cursor = handle.cursor_at(1);
let first = cursor.stream_bytes(2)?.next().transpose()?.unwrap();
assert_eq!(first, b"12");
assert_eq!(cursor.tell(), 3);

## Laziness

In [ ]:
use yggdryl::io::IOBase;
use yggdryl::{IOKind, local};

let path = std::env::temp_dir().join("yggdryl-docs-io-lazy.csv");
let _ = std::fs::remove_file(&path);

// Constructing touches nothing: no file is created, opened, or mapped.
let mut handle = local::File::new(&path)?;
assert!(!handle.exists());

// Reading something absent yields nothing rather than failing.
assert_eq!(handle.size(), 0);
let mut probe = [0_u8; 8];
assert_eq!(handle.pread(0, &mut probe)?, 0);
assert_eq!(handle.kind(), IOKind::Unknown);

// Writing creates the resource, and any parent it needs.
handle.write_all_bytes(b"symbol,price\n")?;
assert_eq!(handle.kind(), IOKind::File);
assert_eq!(handle.read_all_bytes()?, b"symbol,price\n");

handle.close()?;
// Teardown through the abstraction: absence is a no-op success.
handle.remove(false)?;

## Kinds

In [ ]:
use yggdryl::io::{Buffer, IOBase};
use yggdryl::{IOKind, local};

assert_eq!(Buffer::new().kind(), IOKind::Memory);
assert!(IOKind::Memory.is_leaf());

let folder = local::Folder::new(std::env::temp_dir())?;
assert_eq!(folder.kind(), IOKind::Directory);
assert!(folder.is_container());

// Nothing is there, so nothing has decided; a write settles it.
let absent = local::File::new(std::env::temp_dir().join("yggdryl-docs-io-absent.bin"))?;
assert_eq!(absent.kind(), IOKind::Unknown);
assert!(!absent.kind().is_known());

## Bytes or rows

In [ ]:
use yggdryl::io::{Buffer, IOBase};
use yggdryl::{MimeType, local};

// A leaf answers from its representation, and the two are complements.
let mut notes = Buffer::new();
notes.set_media_type(MimeType::PLAIN_TEXT.into());
assert!(notes.is_atomic());
assert!(!notes.is_tabular());

// The name is enough: nothing has been written to this location yet.
let trades = local::File::new(std::env::temp_dir().join("yggdryl-docs-shape.parquet"))?;
assert!(trades.is_tabular());
assert!(!trades.is_atomic());

// A container is neither one whole byte value nor - with nothing under
// it - a table.
let folder = local::Folder::new(std::env::temp_dir())?;
assert!(!folder.is_atomic());

## Whole values

In [ ]:
use yggdryl::io::{Buffer, IOBase};

let mut handle = Buffer::new();
handle.write_all_bytes(b"symbol,price\n")?;

// `append_bytes` reports the offset the bytes landed at.
assert_eq!(handle.append_bytes(b"AAPL,1\n")?, 13);
assert_eq!(handle.read_range_bytes(0, 6)?, b"symbol");
// A range past the end yields what exists rather than failing.
assert!(handle.read_range_bytes(100, 4)?.is_empty());
assert_eq!(handle.read_all_bytes()?.len(), 20);

## Structured values

In [ ]:
use yggdryl::io::{Buffer, IOBase};
use yggdryl::{Field, Url, Scalar};

let media = Url::from_str("file:///trade.json.gz")?.media_type();
let mut handle = Buffer::new().with_media_type(media);
let value = Scalar::from_record([
    ("quantity", Scalar::I64(2)),
    ("symbol", Scalar::from("AAPL")),
])?;
handle.write_scalar(&value)?;

let field = Field::from_str(
    "trade: struct<quantity: int32 not null, symbol: utf8 not null> not null",
)?;
assert_eq!(handle.read_scalar(Some(&field))?[0], Scalar::I64(2));

## Streaming adapters

In [ ]:
use std::io::{Read, Write};

use yggdryl::io::{Buffer, IOBase};

let mut handle = Buffer::new();
handle.writer_at(0).write_all(b"symbol,price\n")?;
handle.append_bytes(b"AAPL,1\n")?;

let mut text = String::new();
handle.reader_at(13).read_to_string(&mut text)?;
assert_eq!(text, "AAPL,1\n");

## Cursors

In [ ]:
use std::io::Read;

use yggdryl::io::{Buffer, IOBase, IOCursor};

let mut cursor = Buffer::new().cursor();
cursor.write_next(b"symbol,price\n")?;
assert_eq!(cursor.tell(), 13);

cursor.seek_to(7);
let mut word = [0_u8; 5];
cursor.read_exact(&mut word)?; // std::io::Read rides the same position
assert_eq!(&word, b"price");

## What the bytes are

In [ ]:
use yggdryl::io::{Buffer, IOBase};
use yggdryl::MimeType;

// Nothing names an in-memory buffer, so its type comes from its bytes.
let mut handle = Buffer::from_bytes(br#"{"symbol":"AAPL"}"#.to_vec());
assert_eq!(handle.media_type().base(), &MimeType::JSON);

// It is re-derived after the content changes.
handle.write_all_bytes(b"PAR1payload")?;
assert_eq!(handle.media_type().base(), &MimeType::PARQUET);

In [ ]:
use yggdryl::io::{Buffer, IOBase};
use yggdryl::{Codec, MimeType, Url};

// A declared type wins, and the codings it carries are what `codec` reports.
let named = Buffer::new().with_media_type(Url::from_str("file:///trades.json.gz")?.media_type());
assert_eq!(named.media_type().base(), &MimeType::JSON);
assert_eq!(named.codec(), Codec::Gzip);

## Adding and removing a coding

In [ ]:
use yggdryl::io::{Buffer, IOBase};
use yggdryl::{Codec, Url};

let mut plain = Buffer::new().with_media_type(Url::from_str("file:///rows.json")?.media_type());
plain.write_all_bytes(br#"{"symbol":"AAPL"}"#)?;
// Nothing wraps these bytes, so there is nothing to undo.
assert_eq!(plain.codec(), Codec::Identity);

let mut encoded =
    Buffer::new().with_media_type(Url::from_str("file:///rows.json.gz")?.media_type());
assert_eq!(encoded.codec(), Codec::Gzip);

// The coding is an argument here, and the target's name is one place to read it from.
let codec = encoded.codec();
plain.compress_into(&mut encoded, codec)?;
assert_eq!(&encoded.read_all_bytes()?[..2], b"\x1f\x8b");

let mut decoded = Buffer::new();
encoded.decompress_into(&mut decoded)?;
assert_eq!(decoded.read_all_bytes()?, plain.read_all_bytes()?);
assert_eq!(decoded.codec(), Codec::Identity);

## Open and close

In [ ]:
use yggdryl::io::{Buffer, Coded, IOBase};
use yggdryl::Codec;

let mut handle = Coded::new(Buffer::new(), Codec::Zstd);
assert!(!handle.opened());

handle.open()?;
assert!(handle.opened());
handle.write_all_bytes(b"symbol,price\n")?;

// Closing publishes the pending write and releases the cache.
handle.close()?;
assert!(!handle.opened());

// The handle stays usable; the next read re-materializes.
assert_eq!(handle.read_all_bytes()?, b"symbol,price\n");

## Clearing and removing

In [ ]:
use yggdryl::io::{Buffer, IOBase};
use yggdryl::local::Folder;

let root = std::env::temp_dir().join(format!("yggdryl-docs-lifecycle-{}", std::process::id()));
let mut folder = Folder::new(&root)?;
folder.truncate(0)?;
folder.child_by_path("a.log")?.write_all_bytes(b"line\n")?;

// Clearing empties the container and keeps it.
folder.clear()?;
assert_eq!(folder.ls(true, false).count(), 0);
assert_eq!(folder.kind(), yggdryl::IOKind::Directory);

// Removing deletes it; a second call succeeds, having done nothing. A
// handle asked for as a container keeps answering `Directory`, because that
// is what it was asked for - the parent's listing is what shows it gone.
let leaf = root.join("nested");
let mut nested = Folder::new(&leaf)?;
nested.truncate(0)?;
assert_eq!(folder.ls(false, false).count(), 1);
nested.remove(false)?;
nested.remove(false)?;
assert_eq!(folder.ls(false, false).count(), 0);
folder.remove(false)?;

// A wrapping handle removes what it wraps, cache included.
let mut coded = yggdryl::gzip::Gzip::new(Buffer::new());
coded.write_all_bytes(b"symbol,price\n")?;
coded.remove(false)?;
assert_eq!(coded.size(), 0);

## Buffer

In [ ]:
use yggdryl::io::{Buffer, IOBase};
use yggdryl::MimeType;

let mut handle = Buffer::with_capacity(1_024);
handle.reserve(4_096)?;
assert!(handle.capacity() >= 4_096);
// Reserving changes the allocation, never the length.
assert_eq!(handle.size(), 0);

handle.pwrite(0, b"symbol,price\n")?;
assert_eq!(handle.as_slice(), b"symbol,price\n");

// A format the bytes cannot identify is declared rather than guessed.
let csv = Buffer::from_bytes(handle.into_bytes()).with_media_type(MimeType::CSV.into());
assert_eq!(csv.media_type().base(), &MimeType::CSV);

## Coded

In [ ]:
use yggdryl::io::{Buffer, Coded, IOBase};
use yggdryl::{Codec, Level, MimeType, Url};

let inner = Buffer::new().with_media_type(Url::from_str("file:///trades.arrows.gz")?.media_type());
let mut handle = Coded::new(inner, Codec::Gzip).with_level(Level::BEST);

// The wrapper's bytes are decoded, so its media type has the coding removed.
assert_eq!(handle.media_type().base(), &MimeType::ARROW_STREAM);
assert_eq!(handle.media_type().encoding_len(), 0);

let payload = "symbol,price\n".repeat(64).into_bytes();
handle.write_all_bytes(&payload)?;
handle.flush()?;

// Reads decompress; the wrapped handle only ever holds the encoded form.
assert_eq!(handle.read_all_bytes()?, payload);
assert!(handle.handle().size() < payload.len() as u64);

## Buffered

In [ ]:
use yggdryl::buffered::BufferedOptions;
use yggdryl::io::{Buffer, IOBase};

let handle = Buffer::from_bytes(vec![4_u8; 4_096]).buffered(BufferedOptions::default());

// The first read fetches the page holding the range; the second is memory.
assert_eq!(handle.read_range_bytes(0, 8)?, [4_u8; 8]);
assert_eq!(handle.read_range_bytes(2_000, 8)?, [4_u8; 8]);
assert_eq!(handle.cached_pages(), 1);

## Roles

In [ ]:
use yggdryl::io::IOBase;
use yggdryl::{IOKind, MimeType, local};

let path = std::env::temp_dir().join("yggdryl-docs-io-folder");
let _ = std::fs::remove_dir_all(&path);
let mut folder = local::Folder::new(&path)?;

// A container holds no bytes: reads are empty, byte writes are refused.
let mut probe = [0_u8; 4];
assert_eq!(folder.pread(0, &mut probe)?, 0);
assert_eq!(folder.size(), 0);
let refused = folder.pwrite(0, b"x").unwrap_err().to_string();
assert!(refused.contains("got the directory"), "{refused}");

// Truncating to zero is the write that brings a container into being.
folder.truncate(0)?;
assert!(folder.exists());
assert_eq!(folder.kind(), IOKind::Directory);
assert_eq!(folder.media_type().base(), &MimeType::DIRECTORY);
assert_eq!(folder.ls(false, false).count(), 0);

std::fs::remove_dir_all(&path)?;

In [ ]:
use yggdryl::io::IOBase;
use yggdryl::{IOKind, local};

// A location that arrived from outside answers by looking at what is there.
let existing = local::Path::new(std::env::temp_dir())?;
assert_eq!(existing.kind(), IOKind::Directory);

let undecided = local::Path::new(std::env::temp_dir().join("yggdryl-docs-io-undecided"))?;
assert_eq!(undecided.kind(), IOKind::Unknown);
assert!(undecided.read_all_bytes()?.is_empty());

// A leaf is not a container: it lists nothing and resolves no child.
let leaf = local::File::new(std::env::temp_dir().join("yggdryl-docs-io-leaf.arrows"))?;
assert_eq!(leaf.ls(true, false).count(), 0);
assert!(leaf.child_by_path("nested").is_err());

## Delegating to a wrapped handle

In [ ]:
use yggdryl::io::{Buffer, IOBase, IOMedia};

/// A wrapper mirrors the handle's bytes rather than owning bytes of its own.
struct Wrapped {
    handle: Buffer,
}

impl IOMedia for Wrapped {
    yggdryl::delegate_iomedia!(handle);
}

impl IOBase for Wrapped {
    yggdryl::delegate_iobase!(handle);
}

fn main() -> Result<(), Box<dyn std::error::Error>> {
    let mut wrapper = Wrapped {
        handle: Buffer::new(),
    };
    wrapper.open()?;
    wrapper.write_all_bytes(b"AAPL")?;

    assert_eq!(wrapper.opened(), wrapper.handle.opened());
    assert_eq!(wrapper.read_all_bytes()?, b"AAPL");
    assert_eq!(wrapper.handle.as_slice(), b"AAPL");
    Ok(())
}

## Arrow batches

In [ ]:
use std::sync::Arc;

use arrow_array::{Int64Array, RecordBatch, StringArray};
use yggdryl::arrow;
use yggdryl::io::{Buffer, IOBase, IOMedia};
use yggdryl::{DataType, Url};

// A non-null struct Field is the schema.
let schema = DataType::from_fields([
    DataType::Int64.required_field("id"),
    DataType::Utf8.nullable_field("symbol"),
])?
.required_field("row");

let arrow_schema = schema.clone().into_arrow_schema()?;
let batch = RecordBatch::try_new(
    Arc::clone(&arrow_schema),
    vec![
        Arc::new(Int64Array::from(vec![1, 2])),
        Arc::new(StringArray::from(vec![Some("AAPL"), None])),
    ],
)?;

// The handle's own media type picks the encoding; no format argument is passed.
let mut handle = Buffer::new().with_media_type(Url::from_str("file:///trades.arrows")?.media_type());
let options = handle.record_options()?;

// Overwrite takes a batch reader; the method name fixes the write intent.
handle.overwrite_arrow_reader(arrow::batch_reader(arrow_schema, [batch]), &options)?;
assert_eq!(handle.read_arrow_field(&options)?, schema);
assert_eq!((handle.row_size()?, handle.column_size()?), (2, 2));

// The read path returns one. Batches arrive one at a time, never as a vector.
let mut rows = 0;
for batch in handle.read_arrow_reader(&options)? {
    rows += batch?.num_rows();
}
assert_eq!(rows, 2);

### Canonical record-write signatures

In [ ]:
use yggdryl::generic::IORecordOptions;
use yggdryl::io::{Buffer, IOBase, IOMedia};
use yggdryl::{DataType, MimeType, Scalar};

struct Quote(i32, &'static str);

impl From<Quote> for Scalar {
    fn from(row: Quote) -> Self {
        Scalar::from_sequence([Scalar::from(row.0), Scalar::from(row.1)])
    }
}

let field = DataType::from_fields([
    DataType::Int32.required_field("id"),
    DataType::Utf8.required_field("symbol"),
])?
.required_field("quote");
let mut handle = Buffer::new().with_media_type(MimeType::ARROW_STREAM.into());
let options = handle.record_options()?.with_field(field);

handle.overwrite_records([Quote(1, "AAPL"), Quote(2, "MSFT")], &options)?;
handle.append_records([Quote(3, "AMD")], &options)?;
handle.merge_records(
    [Quote(2, "MSFT.O")],
    &options.clone().with_merge_by_names(["id"]),
)?;
assert_eq!(
    handle
        .read_arrow_reader(&options)?
        .map(|batch| batch.unwrap().num_rows())
        .sum::<usize>(),
    3,
);

In [ ]:
use yggdryl::io::{Buffer, IOBase, IOMedia};
use yggdryl::MimeType;

// An absent resource holds no batches rather than failing to parse.
let empty = Buffer::new().with_media_type(MimeType::ARROW_STREAM.into());
assert_eq!(
    empty.read_arrow_reader(&empty.record_options()?)?.count(),
    0
);

// An encoding this build does not implement is named rather than guessed.
let csv = Buffer::new().with_media_type(MimeType::CSV.into());
let message = csv.record_options().unwrap_err().to_string();
assert!(message.contains("text/csv"), "{message}");

## Column pushdown

In [ ]:
use std::sync::Arc;

use arrow_array::{Int64Array, RecordBatch, RecordBatchReader, StringArray};
use yggdryl::arrow;
use yggdryl::generic::IORecordOptions;
use yggdryl::io::{Buffer, IOBase, IOMedia};
use yggdryl::{DataType, MimeType};

let stored = DataType::from_fields([
    DataType::Int64.required_field("id"),
    DataType::Utf8.required_field("symbol"),
    DataType::Utf8.required_field("venue"),
])?
.required_field("row");
let arrow_schema = stored.into_arrow_schema()?;

let batch = RecordBatch::try_new(
    Arc::clone(&arrow_schema),
    vec![
        Arc::new(Int64Array::from(vec![1, 2])),
        Arc::new(StringArray::from(vec!["AAPL", "MSFT"])),
        Arc::new(StringArray::from(vec!["XNAS", "XNAS"])),
    ],
)?;

let mut handle = Buffer::new().with_media_type(MimeType::ARROW_STREAM.into());
let plain = handle.record_options()?;
handle.overwrite_arrow_reader(arrow::batch_reader(arrow_schema, [batch]), &plain)?;

// One of the three columns, declared as this read's schema.
let wanted = DataType::from_fields([DataType::Int64.required_field("id")])?.required_field("row");

let projected = handle.read_arrow_reader(&plain.clone().with_field(wanted))?;
assert_eq!(projected.schema().fields().len(), 1);
assert_eq!(projected.map(|batch| batch.unwrap().num_columns()).sum::<usize>(), 1);

// The resource is unchanged: it still holds all three.
assert_eq!(handle.read_arrow_field(&plain)?.field_len(), 3);

// A column it does not hold cannot be projected out of it, so the encoding
// reads everything and the cast supplies that column as nulls.
let invented = DataType::from_fields([
    DataType::Int64.required_field("id"),
    DataType::Utf8.nullable_field("nowhere"),
])?
.required_field("row");
let widened = handle.read_arrow_reader(&plain.with_field(invented))?;
assert_eq!(widened.schema().fields().len(), 2);
assert_eq!(widened.schema().field(1).name(), "nowhere");

## Limiting a read or a write

In [ ]:
use std::sync::Arc;

use arrow_array::{Int64Array, RecordBatch, RecordBatchReader};
use yggdryl::arrow;
use yggdryl::generic::IORecordOptions;
use yggdryl::io::{Buffer, IOBase, IOMedia};
use yggdryl::{DataType, MimeType};

let schema = DataType::from_fields([DataType::Int64.required_field("id")])?
    .required_field("row");
let arrow_schema = schema.into_arrow_schema()?;
let batch = RecordBatch::try_new(
    Arc::clone(&arrow_schema),
    vec![Arc::new(Int64Array::from_iter_values(0..1_000))],
)?;

let mut handle = Buffer::new().with_media_type(MimeType::ARROW_STREAM.into());
let plain = handle.record_options()?;
handle.overwrite_arrow_reader(arrow::batch_reader(arrow_schema, [batch]), &plain)?;

// Ten result rows, exactly: the batch the bound lands inside is sliced.
let first = handle.read_arrow_reader(&plain.clone().with_max_row_size(10))?;
assert_eq!(first.map(|batch| batch.unwrap().num_rows()).sum::<usize>(), 10);

// Zero is a valid ask: the shaped schema answers, and no batch flows.
let mut none = handle.read_arrow_reader(&plain.clone().with_max_row_size(0))?;
assert_eq!(none.schema().fields().len(), 1);
assert!(none.next().is_none());

// A non-zero byte bound always yields at least one row.
let narrow = handle.read_arrow_reader(&plain.clone().with_max_byte_size(1))?;
assert_eq!(narrow.map(|batch| batch.unwrap().num_rows()).sum::<usize>(), 1);

// A limited write truncates the data the caller offered: three rows land,
// and what the bound cut off is never pulled from the reader.
let mut copy = Buffer::new().with_media_type(MimeType::ARROW_STREAM.into());
copy.overwrite_arrow_reader(
    handle.read_arrow_reader(&plain)?,
    &plain.clone().with_max_row_size(3),
)?;
let kept = copy.read_arrow_reader(&plain)?;
assert_eq!(kept.map(|batch| batch.unwrap().num_rows()).sum::<usize>(), 3);

## Appending and merging

In [ ]:
use std::sync::Arc;

use arrow_array::{Int64Array, RecordBatch, StringArray};
use yggdryl::arrow;
use yggdryl::generic::IORecordOptions;
use yggdryl::io::{Buffer, IOBase, IOMedia};
use yggdryl::{DataType, Url};

let schema = DataType::from_fields([
    DataType::Int64.required_field("id"),
    DataType::Utf8.nullable_field("symbol"),
])?
.required_field("row");
let arrow_schema = schema.clone().into_arrow_schema()?;
let rows = |ids: Vec<i64>, symbols: Vec<&'static str>| {
    let batch = RecordBatch::try_new(
        Arc::clone(&arrow_schema),
        vec![
            Arc::new(Int64Array::from(ids)),
            Arc::new(StringArray::from(symbols)),
        ],
    )
    .expect("a batch matching the root");
    arrow::batch_reader(batch.schema(), [batch])
};

let mut handle =
    Buffer::new().with_media_type(Url::from_str("file:///trades.arrows")?.media_type());
let options = handle.record_options()?.with_field(schema.clone());

// Overwrite replaces the resource.
handle.overwrite_arrow_reader(rows(vec![1, 2], vec!["AAPL", "MSFT"]), &options)?;

// Appending reads what is there, chains the new batches after it, and rewrites.
handle.append_arrow_reader(rows(vec![3], vec!["NVDA"]), &options)?;
let total: usize = handle
    .read_arrow_reader(&options)?
    .map(|batch| batch.unwrap().num_rows())
    .sum();
assert_eq!(total, 3);

// Merge requires a match key: `2` updates and `9` appends.
let merging = options.clone().with_merge_by_names(["id"]);
handle.merge_arrow_reader(rows(vec![2, 9], vec!["MSFT.O", "AMD"]), &merging)?;
let total: usize = handle
    .read_arrow_reader(&options)?
    .map(|batch| batch.unwrap().num_rows())
    .sum();
assert_eq!(total, 4);

In [ ]:
use yggdryl::generic::IORecordOptions;
use yggdryl::io::{Buffer, IOBase, IOMedia};
use yggdryl::{arrow, DataType, MimeType};

use arrow_array::{Int64Array, RecordBatch, StringArray};
use std::sync::Arc;
let schema = DataType::from_fields([
    DataType::Int64.required_field("id"),
    DataType::Utf8.nullable_field("symbol"),
])?
.required_field("row");
let batch = RecordBatch::try_new(
    schema.into_arrow_schema()?,
    vec![
        Arc::new(Int64Array::from(vec![1_i64, 2])),
        Arc::new(StringArray::from(vec![Some("AAPL"), Some("MSFT")])),
    ],
)?;

let mut handle = Buffer::new().with_media_type(MimeType::ARROW_STREAM.into());
let options = handle.record_options()?;
handle.overwrite_arrow_reader(arrow::batch_reader(batch.schema(), [batch]), &options)?;

// A read narrowed to one column yields one column.
let selecting = options.with_select_by_names(["symbol"]);
let first = handle.read_arrow_reader(&selecting)?.next().unwrap()?;
assert_eq!(first.num_columns(), 1);
assert_eq!(first.schema().field(0).name(), "symbol");

## Text records

In [ ]:
use yggdryl::io::{Buffer, IOBase};
use yggdryl::text::LineSep;
use yggdryl::Url;

let mut handle = Buffer::new()
    .with_media_type(Url::from_str("file:///trades.jsonl.gz")?.media_type())
    .into_text();
handle.write_all_bytes(&yggdryl::gzip::dump(b"{\"id\":1}\r\n{\"id\":2}\n")?)?;

// A record borrows the window; `text()` is the checked view of its bytes.
let mut records = handle.read_lines()?;
let mut seen: Vec<String> = Vec::new();
while let Some(record) = records.next() {
    seen.push(record?.text()?.to_owned());
}
assert_eq!(seen, ["{\"id\":1}", "{\"id\":2}"]);

// Pinned, a lone `\n` is content rather than a break.
let mixed = Buffer::from_bytes(b"lf\ncrlf\r\nlast".to_vec())
    .into_text()
    .with_linesep(LineSep::CRLF);
let mut pinned = mixed.read_lines()?;
assert_eq!(pinned.next().unwrap()?.text()?, "lf\ncrlf");
assert_eq!(pinned.next().unwrap()?.text()?, "last");

In [ ]:
use yggdryl::io::{Buffer, IOBase};
use yggdryl::text::TextLineOptions;

let handle = Buffer::from_bytes(
    b"2024-02-01 10:00:00.000_000 [ee] [alpha] boom\n  at frame one\n\
      2024-02-01 10:00:01.000_000 [ii] [beta] fine\n".to_vec(),
)
.into_text_with(TextLineOptions::for_logs());

let mut records = handle.read_lines()?;
let first = records.next().unwrap()?;
// The stack frame joined its entry rather than becoming a record.
assert_eq!(first.line_count(), 2);
assert_eq!(first.text()?, "2024-02-01 10:00:00.000_000 [ee] [alpha] boom\n  at frame one");
assert_eq!(records.next().unwrap()?.line_count(), 1);
assert!(records.next().is_none());

### Writing records

In [ ]:
use yggdryl::io::{Buffer, IOBase};
use yggdryl::text::LineSep;

let mut handle = Buffer::new().into_text();
// A lazy iterator, never a collected `Vec`.
handle.write_lines((0..1_000).map(|index| format!("row-{index}")))?;
handle.append_lines(["tail"])?;
assert!(handle.read_all_bytes()?.ends_with(b"row-999\ntail\n"));

// A pinned terminator is written verbatim and read back exactly.
let mut pinned = Buffer::new().into_text().with_linesep(LineSep::CRLF);
pinned.write_lines(["one", "two"])?;
assert_eq!(pinned.read_all_bytes()?, b"one\r\ntwo\r\n");

### Records as Arrow batches

In [ ]:
use yggdryl::io::{Buffer, IOBase};
use yggdryl::text::TextLineOptions;
use yggdryl::{Url, Scalar};

let mut handle = Buffer::new()
    .with_media_type(Url::from_str("file:///app.log.gz")?.media_type());
handle.write_all_bytes(&yggdryl::gzip::dump(
    b"2024-02-01 10:00:00.000_000 [ee] [alpha] boom\n    at frame one\n\
      2024-02-01 10:00:01.500 [ii] [beta] fill 100 @ 187.23\n",
)?)?;

let options = TextLineOptions::with_pattern(
    r"^\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2}\S* \[(?<level>[^\]]+)\] \[(?<logger>[^\]]+)\]",
)?
.try_with_custom_fields([("venue", Scalar::from("XNAS"))])?;

let batches: Vec<_> = handle.read_arrow_lines(&options)?.collect::<Result<_, _>>()?;
assert_eq!(batches.len(), 1);
assert_eq!(batches[0].num_rows(), 2);

let names: Vec<&str> = batches[0].schema_ref().fields().iter().map(|f| f.name().as_str()).collect();
assert_eq!(
    names,
    ["url", "rownum", "date", "time", "unix", "hash", "header", "message",
     "offset", "lines", "level", "logger", "venue"],
);
let unix = batches[0].column(4).as_any().downcast_ref::<arrow_array::Int64Array>().unwrap();
assert_eq!(unix.value(0), 1_706_781_600_000_000_000);
let levels = batches[0].column(10).as_any().downcast_ref::<arrow_array::StringArray>().unwrap();
assert_eq!(levels.value(0), "ee");

### A reader is a configuration document

In [ ]:
use yggdryl::io::{Buffer, IOBase};
use yggdryl::text::TextLineOptions;
use yggdryl::Field;

let document = r#"
pattern: '^(?<stamp>\S+) \[(?<level>[A-Z]+)\]'
byte_size: 1048576
batch_size: 4096
timestamp_capture: stamp
timezone: Europe/Paris
capture_types:
  level: utf8
custom_fields:
  source: gateway
"#;

let options = TextLineOptions::from_value(yggdryl::yaml::from_utf8(document)?)?;
assert_eq!(options.byte_size(), Some(1 << 20));
assert_eq!(options.timestamp_capture(), Some("stamp"));

// The schema answers from the document alone - no resource in sight - so
// the table exists before the first log line does.
let names: Vec<&str> = options.field().fields().iter().map(Field::name).collect();
assert_eq!(names[names.len() - 2..], ["level", "source"]);

let handle = Buffer::from_bytes(b"2024-02-01T10:00:00 [ERROR] boom\n".to_vec());
let batch = handle.read_arrow_lines(&options)?.next().unwrap()?;
assert_eq!(batch.num_rows(), 1);

// And the options project back out, so a reader can be persisted as one.
assert_eq!(TextLineOptions::from_value(options.into_value())?.pattern(), options.pattern());

In [ ]:
use arrow_array::Array;
use yggdryl::io::{Buffer, IOBase};
use yggdryl::text::TextLineOptions;
use yggdryl::{DataType, Url};

let pattern = r"^\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2} \[(?<thread_id>\d+)\] \((?<log_level>\w+)\) qty=(?<qty>[0-9.]+)";

// The standalone builder answers the emitted root without a reader:
// `thread_id` types itself off its own `\d+` sub-pattern.
let schema = TextLineOptions::with_pattern(pattern)?.into_field();
assert_eq!(
    schema.get_field_by_path("thread_id").unwrap().dtype(),
    &DataType::Int64
);

let mut handle = Buffer::new()
    .with_media_type(Url::from_str("file:///app.log")?.media_type());
handle.write_all_bytes(b"2024-02-01 10:00:00 [42] (info) qty=1.50 fill\n")?;

// A declaration types what inference cannot: `qty` lands as a decimal.
let options = TextLineOptions::with_pattern(pattern)?
    .try_with_capture_types([("qty", DataType::decimal(9, 2)?)])?;
let batch = handle.read_arrow_lines(&options)?.next().unwrap()?;
let threads = batch
    .column_by_name("thread_id")
    .unwrap()
    .as_any()
    .downcast_ref::<arrow_array::Int64Array>()
    .unwrap();
assert_eq!(threads.value(0), 42);
let quantities = batch
    .column_by_name("qty")
    .unwrap()
    .as_any()
    .downcast_ref::<arrow_array::Decimal128Array>()
    .unwrap();
assert_eq!(quantities.value(0), 150, "1.50 at scale 2");

### Trimming, and a zone that makes `unix` an instant

In [ ]:
use yggdryl::io::{Buffer, IOBase};
use yggdryl::text::{Strip, TextLineOptions};
use yggdryl::Timezone;

let handle = Buffer::from_bytes(b"2024-02-01T00:00:00 [INFO] x\n".to_vec());
let pattern = r"^(?<stamp>\S+) \[(?<level>[A-Z]+)\]";

let naive = TextLineOptions::with_pattern(pattern)?;
let batch = handle.read_arrow_lines(&naive)?.next().unwrap()?;
let unix = |batch: &arrow_array::RecordBatch| {
    use arrow_array::Array as _;
    batch.column(4).as_any().downcast_ref::<arrow_array::Int64Array>().unwrap().value(0)
};
assert_eq!(unix(&batch), 1_706_745_600_000_000_000);

// Read in Paris, the same civil reading is an hour earlier as an instant.
let zoned = TextLineOptions::with_pattern(pattern)?
    .try_with_timezone("Europe/Paris".parse::<Timezone>()?)?
    .with_rstrip(Strip::Ascii);
let batch = handle.read_arrow_lines(&zoned)?.next().unwrap()?;
assert_eq!(unix(&batch), 1_706_745_600_000_000_000 - 3_600 * 1_000_000_000);

### Streaming a trading log into a partitioned Iceberg table

In [ ]:
use yggdryl::iceberg::Catalog;
use yggdryl::io::IOBase;
use yggdryl::local::Folder;
use yggdryl::text::TextLineOptions;
use yggdryl::Scalar;

let root = std::env::temp_dir().join("yggdryl-doc-log-lake");
let _ = std::fs::remove_dir_all(&root);
let logs = root.join("incoming");
std::fs::create_dir_all(&logs)?;
std::fs::write(
    logs.join("a.log"),
    b"2024-02-01 10:00:00.000_000 [ee] [alpha] boom\n    at frame one\n\
      2024-02-01 10:00:01.000_000 [ii] [beta] fill 100 @ 187.23\n",
)?;
std::fs::write(
    logs.join("b.log.gz"),
    yggdryl::gzip::dump(b"2024-02-01 11:00:00.000_000 [ii] [gamma] fill 200 @ 188.01\n")?,
)?;

let options = TextLineOptions::with_pattern(
    r"^\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2}\S* \[(?<level>[^\]]+)\] \[(?<logger>[^\]]+)\]",
)?
.try_with_custom_fields([("venue", Scalar::from("XNAS"))])?;

// The options already know the schema, so the table exists - partitioned
// by `level` - before the first log line is parsed.
let catalog = Catalog::new(Folder::new(root.join("warehouse"))?);
catalog
    .tables()
    .create("logs.app", options.field().with_partition_fields(&["level"])?)?;

// The append consumes the parse itself: a lazy reader over both leaves,
// the gzip one decoded as a stream, routed row by row into `level=...`
// partitions and committed as one snapshot.
let table = catalog
    .tables()
    .append_arrow_reader("logs.app", Folder::new(&logs)?.into_arrow_lines(&options)?)?;

let snapshot = table.current_snapshot().expect("one commit");
assert_eq!(snapshot.operation(), "append");
assert_eq!(snapshot.summary_value("added-records"), Some("3"));
assert_eq!(snapshot.summary_value("added-data-files"), Some("2"), "one file per level");

// A second day appends a second snapshot; the metadata accumulates.
let mut day_two = yggdryl::io::Buffer::new()
    .with_media_type(yggdryl::Url::from_str("file:///c.log")?.media_type());
day_two.write_all_bytes(b"2024-02-02 09:30:00.000_000 [ee] [delta] second day\n")?;
let table = catalog
    .tables()
    .append_arrow_reader("logs.app", day_two.into_arrow_lines(&options)?)?;
assert_eq!(table.metadata().snapshots().len(), 2);
assert_eq!(table.current_snapshot().unwrap().summary_value("total-records"), Some("4"));
assert_eq!(table.manifests()?.len(), 2, "one manifest per append");

// The partition tuples just committed are what the next read prunes with.
let plan = table.plan(&[("level", "ee")])?;
assert_eq!(plan.tasks.len(), 2);
assert_eq!(plan.files_skipped(), 1, "the ii file is never opened");
let rows: usize = table
    .scan_where(&[("level", "ii")], None)?
    .map(|batch| batch.map(|b| b.num_rows()))
    .sum::<Result<usize, _>>()?;
assert_eq!(rows, 2);

let _ = std::fs::remove_dir_all(&root);

### The full pipeline, small enough to run

In [ ]:
use arrow_array::{Array, Int64Array};
use yggdryl::iceberg::Catalog;
use yggdryl::io::IOBase;
use yggdryl::local::Folder;
use yggdryl::text::TextLineOptions;
use yggdryl::Scalar;

let pattern = r"^\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2}\S* \[(?<level>[^\]]+)\] \[(?<logger>[^\]]+)\] \[(?<thread_id>\d+)\] took=(?<latency_us>\d+)";
// The older extractor: same records, no thread column.
let archived = r"^\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2}\S* \[(?<level>[^\]]+)\] \[(?<logger>[^\]]+)\] \[\d+\] took=(?<latency_us>\d+)";

let root = std::env::temp_dir().join(format!("yggdryl-docs-pipeline-{}", std::process::id()));
let _ = std::fs::remove_dir_all(&root);
let incoming = root.join("incoming");
let archive = root.join("archive");
std::fs::create_dir_all(&incoming)?;
std::fs::create_dir_all(&archive)?;

// Three rotated leaves in three codings; the second record spans a stack trace.
let first = concat!(
    "2024-02-01 10:00:00.000000 [ii] [engine] [3] took=120 fill 100 SYMB-0001\n",
    "2024-02-01 10:00:01.000000 [ee] [engine] [4] took=980 fill 101 SYMB-0002\n",
    "    at engine::match(order.rs:118)\n",
    "    at engine::step(order.rs:64)\n",
    "2024-02-01 10:00:02.000000 [ww] [router] [5] took=240 fill 102 SYMB-0003\n",
);
std::fs::write(incoming.join("app-0.log.gz"), yggdryl::gzip::dump(first.as_bytes())?)?;
std::fs::write(
    incoming.join("app-1.log"),
    b"2024-02-01 10:00:03.000000 [ee] [ledger] [6] took=770 fill 103 SYMB-0004\n",
)?;
std::fs::write(
    incoming.join("app-2.log.zst"),
    yggdryl::zstd::dump(b"2024-02-01 10:00:04.000000 [ii] [feed] [7] took=100 fill 104 SYMB-0005\n")?,
)?;
std::fs::write(
    archive.join("app-9.log.gz"),
    yggdryl::gzip::dump(b"2024-01-31 23:59:59.000000 [ee] [engine] [2] took=310 fill 099 SYMB-0000\n")?,
)?;

// 1. The extractor, and the table it implies - both before anything is read.
let options = TextLineOptions::with_pattern(pattern)?
    .try_with_custom_fields([("source", Scalar::from("gateway"))])?
    .with_byte_size(8 * 1024 * 1024);
let marked = options.field().with_partition_fields(&["level"])?;

let catalog = Catalog::new(Folder::new(root.join("warehouse"))?);
let mut table = catalog.tables().create("logs.app", marked)?;

// 2, 3, 4. One handle per folder, and one lazy combine over the two.
let older = TextLineOptions::with_pattern(archived)?
    .try_with_custom_fields([("source", Scalar::from("archive"))])?;
let stream = yggdryl::arrow::combined(
    Folder::new(&incoming)?.into_arrow_lines(&options)?,
    Folder::new(&archive)?.into_arrow_lines(&older)?,
)?;

// 5. One commit, handed the reader itself - never a Vec of batches.
table.commit_append(stream)?;

// The read-back asserts on the table, not on anything held in memory.
let mut rows = 0_usize;
let mut latency = 0_i64;
let mut threadless = 0_usize;
for batch in table.scan(None)? {
    let batch = batch?;
    rows += batch.num_rows();
    let took = batch
        .column_by_name("latency_us")
        .expect("the typed capture")
        .as_any()
        .downcast_ref::<Int64Array>()
        .expect("int64 by inference, not by declaration");
    latency += (0..batch.num_rows()).map(|row| took.value(row)).sum::<i64>();
    threadless += batch
        .column_by_name("thread_id")
        .expect("the merged column")
        .null_count();
}
// Five live records - not the seven lines they occupy - and one archived.
assert_eq!(rows, 6);
assert_eq!(latency, 120 + 980 + 240 + 770 + 100 + 310);
assert_eq!(threadless, 1, "the archived extractor had no thread column");

let reopened = catalog.table("logs.app")?;
assert_eq!(reopened.metadata().default_spec()?.fields[0].name, "level");
assert!(reopened.schema()?.get_field_by_path("source").is_some());

let _ = std::fs::remove_dir_all(&root);

## Globbing and Hive partitions

In [ ]:
use yggdryl::io::IOBase;
use yggdryl::local::Folder;

let root = std::env::temp_dir().join("yggdryl-doc-lake");
let _ = std::fs::remove_dir_all(&root);
for year in ["2024", "2025"] {
    let leaf = root.join(format!("year={year}")).join("month=01");
    std::fs::create_dir_all(&leaf)?;
    std::fs::write(leaf.join("part-0.parquet"), b"parquet")?;
}

let lake = Folder::new(&root)?;

// A fixed prefix is descended, not listed and filtered.
assert_eq!(lake.glob("year=2024/**/*.parquet", false)?.count(), 1);
assert_eq!(lake.glob("**/*.parquet", false)?.count(), 2);

// Partition filters select the leaves to overwrite or upsert.
let selected: Vec<_> = lake
    .children_where(&[("year", "2024")], false)?
    .collect::<yggdryl::Result<_>>()?;
assert_eq!(selected.len(), 1);
assert_eq!(selected[0].partitions(), vec![
    ("year".to_owned(), "2024".to_owned()),
    ("month".to_owned(), "01".to_owned()),
]);

let _ = std::fs::remove_dir_all(&root);

## Partition pruning and filtering

In [ ]:
use yggdryl::generic::{IORecordOptions, RecordOptions};
use yggdryl::MimeType;

let options = RecordOptions::for_mime_type(&MimeType::ARROW_STREAM)?
    .with_filter_partitions([("year", "2024"), ("month", "01")]);
// handle.read_arrow_reader(&options)? now reads only the January
// 2024 leaves, and only their matching rows.

## Partition columns in the data

In [ ]:
use yggdryl::generic::{Holder, IORecordOptions, RecordOptions};
use yggdryl::io::{IOBase, IOMedia};
use yggdryl::{DataType, MimeType};

let root = std::env::temp_dir().join("yggdryl-doc-partitioned");
let _ = std::fs::remove_dir_all(&root);
std::fs::create_dir_all(root.join("year=2024").join("month=01"))?;

let schema = DataType::from_fields([
    DataType::Int64.required_field("price"),
    DataType::Int32.required_field("year"),
    DataType::Utf8.required_field("month"),
])?
.required_field("row");
let arrow_schema = schema.clone().into_arrow_schema()?;
let batch = arrow_array::RecordBatch::try_new(
    std::sync::Arc::clone(&arrow_schema),
    vec![
        std::sync::Arc::new(arrow_array::Int64Array::from(vec![10, 20])),
        std::sync::Arc::new(arrow_array::Int32Array::from(vec![2024, 2024])),
        std::sync::Arc::new(arrow_array::StringArray::from(vec!["01", "01"])),
    ],
)?;

// The rows carry every column; the write drops the two the path spells out.
let mut lake = Holder::folder(&root)?;
let options = RecordOptions::for_mime_type(&MimeType::ARROW_STREAM)?.with_field(schema.clone());
lake.overwrite_arrow_reader(
    yggdryl::arrow::batch_reader(arrow_schema, [batch]),
    &options,
)?;

// Only `price` reached the leaf; the other two are the directory names.
let leaf = lake.child_by_path("year=2024/month=01/part-0.arrows")?;
assert_eq!(
    leaf.read_arrow_field(&RecordOptions::for_media_type(leaf.media_type())?)?.field_len(),
    1
);

// Reading the folder restores them with their declared types.
let restored = lake
    .read_arrow_reader(&options)?
    .next()
    .expect("one batch")?;
assert_eq!(restored.num_columns(), 3);
assert_eq!(restored.schema().field(1).data_type(), &arrow_schema::DataType::Int32);

let _ = std::fs::remove_dir_all(&root);

In [ ]:
use yggdryl::generic::{Holder, IORecordOptions, RecordOptions};
use yggdryl::io::{IOBase, IOMedia};
use yggdryl::{DataType, MimeType};

let root = std::env::temp_dir().join("yggdryl-doc-declared-layout");
let _ = std::fs::remove_dir_all(&root);
std::fs::create_dir_all(&root)?;

// Nothing is on disk, so nothing spells a layout. The schema does.
let schema = DataType::from_fields([
    DataType::Int64.required_field("price"),
    DataType::Int32.required_field("year"),
])?
.required_field("row")
.with_partition_fields(&["year"])?;
assert_eq!(schema.partition_field_names().collect::<Vec<_>>(), ["year"]);

let arrow_schema = schema.clone().into_arrow_schema()?;
let batch = arrow_array::RecordBatch::try_new(
    std::sync::Arc::clone(&arrow_schema),
    vec![
        std::sync::Arc::new(arrow_array::Int64Array::from(vec![10, 20])),
        std::sync::Arc::new(arrow_array::Int32Array::from(vec![2024, 2024])),
    ],
)?;

let mut lake = Holder::folder(&root)?;
let options = RecordOptions::for_mime_type(&MimeType::ARROW_STREAM)?.with_field(schema);
lake.overwrite_arrow_reader(
    yggdryl::arrow::batch_reader(arrow_schema, [batch]),
    &options,
)?;

// The directory came from the declaration, and the leaf stores what the
// path does not carry.
assert!(root.join("year=2024").is_dir());

// Reading it back reports the layout without being told it.
let derived = lake.read_arrow_field(
    &RecordOptions::for_mime_type(&MimeType::ARROW_STREAM)?,
)?;
assert_eq!(derived.partition_field_names().collect::<Vec<_>>(), ["year"]);

let _ = std::fs::remove_dir_all(&root);